## 📦 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, auc,
    precision_recall_curve, average_precision_score,
    cohen_kappa_score, matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
import tensorflow as tf
from tensorflow import keras
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## 📂 2. Load Models, Data, and Predictions from Module 3

In [ ]:
# Load models
model_binary = keras.models.load_model('/content/best_model_binary.h5')
model_multi = keras.models.load_model('/content/best_model_multiclass.h5')

# Load test data
X_test = np.load('/content/X_test.npy')
y_test_binary = np.load('/content/y_test_binary.npy')
y_test_multi = np.load('/content/y_test_multi.npy')

# Reshape for LSTM
X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# Load predictions
y_pred_binary = np.load('/content/y_pred_binary.npy')
y_pred_binary_prob = np.load('/content/y_pred_binary_prob.npy')
y_pred_multi = np.load('/content/y_pred_multi.npy')
y_pred_multi_prob = np.load('/content/y_pred_multi_prob.npy')

# Load training history
with open('/content/history_binary.pkl', 'rb') as f:
    history_binary = pickle.load(f)
    
with open('/content/history_multi.pkl', 'rb') as f:
    history_multi = pickle.load(f)

print("✅ All Data Loaded Successfully!")
print("="*70)
print(f"📊 Test Set Size: {X_test.shape[0]:,} samples")
print(f"🎯 Binary Labels: {np.unique(y_test_binary)}")
print(f"🎯 Multi-Class Labels: {np.unique(y_test_multi)}")

## 📊 3. Binary Classification - ROC Curve Analysis

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test_binary, y_pred_binary_prob)
roc_auc = roc_auc_score(y_test_binary, y_pred_binary_prob)

# Plot ROC Curve
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Binary Classification (Normal vs Attack)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 ROC-AUC Analysis:")
print("="*70)
print(f"  AUC Score: {roc_auc:.4f}")
print(f"  Interpretation: {'Excellent' if roc_auc > 0.9 else 'Good' if roc_auc > 0.8 else 'Fair' if roc_auc > 0.7 else 'Poor'}")

# Find optimal threshold (Youden's J statistic)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f"\n🎯 Optimal Threshold (Youden's J): {optimal_threshold:.4f}")
print(f"  At this threshold:")
print(f"    - True Positive Rate: {tpr[optimal_idx]:.4f}")
print(f"    - False Positive Rate: {fpr[optimal_idx]:.4f}")

## 📈 4. Binary Classification - Precision-Recall Curve

In [ ]:
# Calculate Precision-Recall curve
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test_binary, y_pred_binary_prob)
avg_precision = average_precision_score(y_test_binary, y_pred_binary_prob)

# Plot Precision-Recall Curve
plt.figure(figsize=(10, 8))
plt.plot(recall_vals, precision_vals, color='blue', lw=3, label=f'PR curve (AP = {avg_precision:.4f})')
plt.axhline(y=y_test_binary.sum()/len(y_test_binary), color='red', linestyle='--', lw=2, label='Baseline (Random)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve - Binary Classification', fontsize=14, fontweight='bold')
plt.legend(loc="lower left", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📈 Precision-Recall Analysis:")
print("="*70)
print(f"  Average Precision (AP): {avg_precision:.4f}")
print(f"  Baseline (Random): {y_test_binary.sum()/len(y_test_binary):.4f}")
print(f"\n📌 Note: PR curves are better for imbalanced datasets")

## 🎯 5. Binary Classification - Threshold Analysis

In [ ]:
# Analyze performance at different thresholds
thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7]

print("🔍 Threshold Analysis - Binary Classification:")
print("="*70)
print(f"{'Threshold':<12} {'Accuracy':<10} {'Precision':<12} {'Recall':<10} {'F1-Score':<10}")
print("="*70)

threshold_results = []
for threshold in thresholds_to_test:
    y_pred_temp = (y_pred_binary_prob > threshold).astype(int).flatten()
    acc = accuracy_score(y_test_binary, y_pred_temp)
    prec = precision_score(y_test_binary, y_pred_temp, zero_division=0)
    rec = recall_score(y_test_binary, y_pred_temp, zero_division=0)
    f1 = f1_score(y_test_binary, y_pred_temp, zero_division=0)
    
    threshold_results.append({
        'threshold': threshold,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1
    })
    
    print(f"{threshold:<12.2f} {acc:<10.4f} {prec:<12.4f} {rec:<10.4f} {f1:<10.4f}")

# Visualize threshold impact
df_thresholds = pd.DataFrame(threshold_results)

plt.figure(figsize=(12, 6))
plt.plot(df_thresholds['threshold'], df_thresholds['accuracy'], 'o-', linewidth=2, label='Accuracy', markersize=8)
plt.plot(df_thresholds['threshold'], df_thresholds['precision'], 's-', linewidth=2, label='Precision', markersize=8)
plt.plot(df_thresholds['threshold'], df_thresholds['recall'], '^-', linewidth=2, label='Recall', markersize=8)
plt.plot(df_thresholds['threshold'], df_thresholds['f1'], 'd-', linewidth=2, label='F1-Score', markersize=8)
plt.xlabel('Classification Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Performance Metrics vs Classification Threshold', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 📊 6. Multi-Class - Per-Class ROC Curves

In [ ]:
# Check actual number of classes in predictions and test data
print("🔍 Analyzing multi-class data dimensions...")
print(f"  y_pred_multi_prob shape: {y_pred_multi_prob.shape}")
print(f"  Unique classes in y_test_multi: {np.unique(y_test_multi)}")
print(f"  Number of unique classes: {len(np.unique(y_test_multi))}")

# Determine actual number of classes from predictions
n_classes_actual = y_pred_multi_prob.shape[1]
n_classes_test = len(np.unique(y_test_multi))

print(f"\n  Model output classes: {n_classes_actual}")
print(f"  Test set classes: {n_classes_test}")

# Use the actual number of classes
n_classes = n_classes_actual
all_class_names = ['DoS', 'Normal', 'Other', 'Probe', 'R2L', 'U2R']
class_names = all_class_names[:n_classes]

print(f"\n  Using {n_classes} classes: {class_names}")

# Binarize based on actual classes present
y_test_bin = label_binarize(y_test_multi, classes=range(n_classes))

# Check for NaN values and handle them
print("\n🔍 Checking for NaN values in predictions...")
print(f"  NaN count in y_pred_multi_prob: {np.isnan(y_pred_multi_prob).sum()}")

# Replace NaN values with 0
if np.isnan(y_pred_multi_prob).any():
    print("  ⚠️ Found NaN values - replacing with 0")
    y_pred_multi_prob_clean = np.nan_to_num(y_pred_multi_prob, nan=0.0)
else:
    y_pred_multi_prob_clean = y_pred_multi_prob

# Compute ROC curve and AUC for each class
fpr_multi = dict()
tpr_multi = dict()
roc_auc_multi = dict()

for i in range(n_classes):
    # Check if this class exists in test set
    if y_test_bin[:, i].sum() > 0:
        try:
            fpr_multi[i], tpr_multi[i], _ = roc_curve(y_test_bin[:, i], y_pred_multi_prob_clean[:, i])
            roc_auc_multi[i] = auc(fpr_multi[i], tpr_multi[i])
        except Exception as e:
            print(f"  ⚠️ Error computing ROC for class {class_names[i]}: {e}")
            fpr_multi[i], tpr_multi[i] = [0, 1], [0, 1]
            roc_auc_multi[i] = 0.5
    else:
        print(f"  ⚠️ Class {class_names[i]} has no positive samples in test set - AUC set to 0.5")
        fpr_multi[i], tpr_multi[i] = [0, 1], [0, 1]
        roc_auc_multi[i] = 0.5

# Plot ROC curves for all classes
plt.figure(figsize=(12, 8))
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown']

for i in range(n_classes):
    color = colors[i % len(colors)]
    plt.plot(fpr_multi[i], tpr_multi[i], color=color, lw=2.5,
             label=f'{class_names[i]} (AUC = {roc_auc_multi[i]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Multi-Class Classification (One-vs-Rest)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Per-Class AUC Scores:")
print("="*70)
for i, class_name in enumerate(class_names):
    print(f"  {class_name:<10}: {roc_auc_multi[i]:.4f}")
print(f"\n  Macro-Average AUC: {np.mean(list(roc_auc_multi.values())):.4f}")

## 🔍 7. Error Analysis - Misclassification Patterns

In [ ]:
# Analyze misclassifications in binary model
print("🔍 BINARY MODEL - Error Analysis:")
print("="*70)

# False Positives (Normal predicted as Attack)
fp_mask = (y_test_binary == 0) & (y_pred_binary == 1)
fp_count = fp_mask.sum()
fp_prob = y_pred_binary_prob[fp_mask].flatten()

print(f"\n❌ False Positives: {fp_count:,} ({fp_count/len(y_test_binary)*100:.2f}%)")
print(f"  Avg Confidence: {fp_prob.mean():.4f}")
print(f"  Min Confidence: {fp_prob.min():.4f}")
print(f"  Max Confidence: {fp_prob.max():.4f}")

# False Negatives (Attack predicted as Normal)
fn_mask = (y_test_binary == 1) & (y_pred_binary == 0)
fn_count = fn_mask.sum()
fn_prob = y_pred_binary_prob[fn_mask].flatten()

print(f"\n❌ False Negatives: {fn_count:,} ({fn_count/len(y_test_binary)*100:.2f}%)")
print(f"  Avg Confidence: {fn_prob.mean():.4f}")
print(f"  Min Confidence: {fn_prob.min():.4f}")
print(f"  Max Confidence: {fn_prob.max():.4f}")

# Visualize error distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# False Positives
axes[0].hist(fp_prob, bins=30, color='coral', alpha=0.7, edgecolor='black')
axes[0].set_title(f'False Positives - Confidence Distribution\n(n={fp_count:,})', fontweight='bold')
axes[0].set_xlabel('Predicted Probability (Attack)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# False Negatives
axes[1].hist(fn_prob, bins=30, color='lightblue', alpha=0.7, edgecolor='black')
axes[1].set_title(f'False Negatives - Confidence Distribution\n(n={fn_count:,})', fontweight='bold')
axes[1].set_xlabel('Predicted Probability (Attack)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 8. Multi-Class - Per-Class Performance Analysis

In [ ]:
# Detailed per-class metrics
from sklearn.metrics import classification_report

print("📊 MULTI-CLASS - Detailed Per-Class Performance:")
print("="*70)

# Get unique classes in test and predictions
unique_test = np.unique(y_test_multi)
unique_pred = np.unique(y_pred_multi)
all_unique = np.unique(np.concatenate([unique_test, unique_pred]))

print(f"\n  Classes in test set: {unique_test}")
print(f"  Classes in predictions: {unique_pred}")
print(f"  All unique classes: {all_unique}")

# Create class names for all classes present in the data
all_class_names_full = ['DoS', 'Normal', 'Other', 'Probe', 'R2L', 'U2R']
class_names_for_report = [all_class_names_full[i] for i in all_unique if i < len(all_class_names_full)]

print(f"  Using class names: {class_names_for_report}")

# Get classification report as dict with explicit labels
report_dict = classification_report(y_test_multi, y_pred_multi, 
                                   labels=all_unique,
                                   target_names=class_names_for_report, 
                                   output_dict=True,
                                   zero_division=0)

# Create DataFrame for better visualization
df_report = pd.DataFrame(report_dict).transpose()
df_report = df_report.iloc[:-3]  # Remove accuracy, macro avg, weighted avg

print(f"\n{df_report.to_string()}")

# Visualize per-class metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['precision', 'recall', 'f1-score']
colors_bar = ['#3498db', '#e74c3c', '#2ecc71']

for idx, (metric, color) in enumerate(zip(metrics, colors_bar)):
    values = df_report[metric].values
    axes[idx].barh(class_names_for_report, values, color=color, alpha=0.7, edgecolor='black')
    axes[idx].set_xlabel(metric.capitalize(), fontsize=12)
    axes[idx].set_xlim([0, 1])
    axes[idx].set_title(f'{metric.capitalize()} by Class', fontweight='bold')
    axes[idx].grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, v in enumerate(values):
        axes[idx].text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# Find best and worst performing classes
best_class_idx = df_report['f1-score'].idxmax()
worst_class_idx = df_report['f1-score'].idxmin()

print(f"\n🏆 Best Performing Class: {best_class_idx} (F1={df_report.loc[best_class_idx, 'f1-score']:.4f})")
print(f"⚠️ Worst Performing Class: {worst_class_idx} (F1={df_report.loc[worst_class_idx, 'f1-score']:.4f})")

## 🎯 9. Advanced Metrics - Cohen's Kappa & Matthews Correlation

In [ ]:
# Calculate advanced metrics for both models
print("🎯 Advanced Statistical Metrics:")
print("="*70)

# Binary Model
kappa_binary = cohen_kappa_score(y_test_binary, y_pred_binary)
mcc_binary = matthews_corrcoef(y_test_binary, y_pred_binary)

print("\n📊 BINARY CLASSIFICATION:")
print(f"  Cohen's Kappa: {kappa_binary:.4f}")
print(f"    Interpretation: ", end="")
if kappa_binary > 0.8:
    print("Almost Perfect Agreement")
elif kappa_binary > 0.6:
    print("Substantial Agreement")
elif kappa_binary > 0.4:
    print("Moderate Agreement")
elif kappa_binary > 0.2:
    print("Fair Agreement")
else:
    print("Slight Agreement")

print(f"\n  Matthews Correlation Coefficient: {mcc_binary:.4f}")
print(f"    Range: -1 (worst) to +1 (best)")
print(f"    Interpretation: ", end="")
if mcc_binary > 0.7:
    print("Strong Positive Correlation")
elif mcc_binary > 0.4:
    print("Moderate Positive Correlation")
elif mcc_binary > 0.1:
    print("Weak Positive Correlation")
else:
    print("Negligible Correlation")

# Multi-Class Model
kappa_multi = cohen_kappa_score(y_test_multi, y_pred_multi)

print("\n📊 MULTI-CLASS CLASSIFICATION:")
print(f"  Cohen's Kappa: {kappa_multi:.4f}")
print(f"    Interpretation: ", end="")
if kappa_multi > 0.8:
    print("Almost Perfect Agreement")
elif kappa_multi > 0.6:
    print("Substantial Agreement")
elif kappa_multi > 0.4:
    print("Moderate Agreement")
elif kappa_multi > 0.2:
    print("Fair Agreement")
else:
    print("Slight Agreement")

print("\n📌 Note: These metrics account for chance agreement, unlike simple accuracy")

## 📈 10. Learning Curves Analysis

In [ ]:
# Analyze training convergence
print("📈 Training Convergence Analysis:")
print("="*70)

# Binary Model
epochs_binary = len(history_binary['loss'])
final_train_acc_binary = history_binary['accuracy'][-1]
final_val_acc_binary = history_binary['val_accuracy'][-1]
final_train_loss_binary = history_binary['loss'][-1]
final_val_loss_binary = history_binary['val_loss'][-1]

print("\n🔵 BINARY MODEL:")
print(f"  Epochs Trained: {epochs_binary}")
print(f"  Final Training Accuracy: {final_train_acc_binary:.4f}")
print(f"  Final Validation Accuracy: {final_val_acc_binary:.4f}")
print(f"  Generalization Gap: {abs(final_train_acc_binary - final_val_acc_binary):.4f}")
print(f"  Final Training Loss: {final_train_loss_binary:.4f}")
print(f"  Final Validation Loss: {final_val_loss_binary:.4f}")

# Multi-Class Model
epochs_multi = len(history_multi['loss'])
final_train_acc_multi = history_multi['accuracy'][-1]
final_val_acc_multi = history_multi['val_accuracy'][-1]
final_train_loss_multi = history_multi['loss'][-1]
final_val_loss_multi = history_multi['val_loss'][-1]

print("\n🟠 MULTI-CLASS MODEL:")
print(f"  Epochs Trained: {epochs_multi}")
print(f"  Final Training Accuracy: {final_train_acc_multi:.4f}")
print(f"  Final Validation Accuracy: {final_val_acc_multi:.4f}")
print(f"  Generalization Gap: {abs(final_train_acc_multi - final_val_acc_multi):.4f}")
print(f"  Final Training Loss: {final_train_loss_multi:.4f}")
print(f"  Final Validation Loss: {final_val_loss_multi:.4f}")

# Check for overfitting
print("\n🔍 Overfitting Assessment:")
if abs(final_train_acc_binary - final_val_acc_binary) > 0.1:
    print("  ⚠️ Binary Model: Possible overfitting detected (gap > 0.1)")
else:
    print("  ✅ Binary Model: Good generalization")
    
if abs(final_train_acc_multi - final_val_acc_multi) > 0.1:
    print("  ⚠️ Multi-Class Model: Possible overfitting detected (gap > 0.1)")
else:
    print("  ✅ Multi-Class Model: Good generalization")

## 🎨 11. Confusion Matrix - Normalized View

In [ ]:
# Normalized confusion matrix for multi-class
cm_multi = confusion_matrix(y_test_multi, y_pred_multi)
cm_multi_norm = cm_multi.astype('float') / cm_multi.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute counts
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix - Absolute Counts', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized percentages
sns.heatmap(cm_multi_norm, annot=True, fmt='.2%', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], cbar_kws={'label': 'Percentage'})
axes[1].set_title('Confusion Matrix - Normalized (Row %)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

# Identify most confused class pairs
print("\n🔄 Most Confused Class Pairs:")
print("="*70)

# Get off-diagonal confusion values
confusion_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm_multi[i, j] > 0:
            confusion_pairs.append({
                'true': class_names[i],
                'predicted': class_names[j],
                'count': cm_multi[i, j],
                'percentage': cm_multi_norm[i, j]
            })

# Sort by count
confusion_pairs_sorted = sorted(confusion_pairs, key=lambda x: x['count'], reverse=True)

for idx, pair in enumerate(confusion_pairs_sorted[:5], 1):
    print(f"  {idx}. {pair['true']:8s} → {pair['predicted']:8s}: {pair['count']:5,} samples ({pair['percentage']:.1%})")

## 📊 12. Model Comparison Summary

In [ ]:
# Create comprehensive comparison
comparison_data = {
    'Metric': [
        'Accuracy',
        'Precision (weighted)',
        'Recall (weighted)',
        'F1-Score (weighted)',
        'ROC-AUC',
        "Cohen's Kappa",
        'MCC',
        'Training Epochs',
        'Parameters'
    ],
    'Binary Model': [
        accuracy_score(y_test_binary, y_pred_binary),
        precision_score(y_test_binary, y_pred_binary),
        recall_score(y_test_binary, y_pred_binary),
        f1_score(y_test_binary, y_pred_binary),
        roc_auc,
        kappa_binary,
        mcc_binary,
        epochs_binary,
        model_binary.count_params()
    ],
    'Multi-Class Model': [
        accuracy_score(y_test_multi, y_pred_multi),
        precision_score(y_test_multi, y_pred_multi, average='weighted'),
        recall_score(y_test_multi, y_pred_multi, average='weighted'),
        f1_score(y_test_multi, y_pred_multi, average='weighted'),
        np.mean(list(roc_auc_multi.values())),
        kappa_multi,
        'N/A',
        epochs_multi,
        model_multi.count_params()
    ]
}

df_comparison = pd.DataFrame(comparison_data)

print("\n📊 MODEL COMPARISON SUMMARY:")
print("="*70)
print(df_comparison.to_string(index=False))

# Visualize key metrics comparison
metrics_to_plot = ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1-Score (weighted)']
binary_values = df_comparison[df_comparison['Metric'].isin(metrics_to_plot)]['Binary Model'].values
multi_values = df_comparison[df_comparison['Metric'].isin(metrics_to_plot)]['Multi-Class Model'].values

x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, binary_values, width, label='Binary Model', color='skyblue', edgecolor='black')
bars2 = ax.bar(x + width/2, multi_values, width, label='Multi-Class Model', color='lightcoral', edgecolor='black')

ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Binary vs Multi-Class Model Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, rotation=15, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1])

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 📋 13. Module 4 Summary Report

In [ ]:
print("\n" + "="*70)
print("📋 MODULE 4: EVALUATION - SUMMARY REPORT")
print("="*70)

print("\n✅ EVALUATION TECHNIQUES APPLIED:")
print("\n1️⃣ ROC-AUC Analysis:")
print(f"   ✓ Binary Model AUC: {roc_auc:.4f}")
print(f"   ✓ Multi-Class Macro-Avg AUC: {np.mean(list(roc_auc_multi.values())):.4f}")

print("\n2️⃣ Precision-Recall Analysis:")
print(f"   ✓ Average Precision: {avg_precision:.4f}")
print("   ✓ Critical for imbalanced datasets")

print("\n3️⃣ Threshold Optimization:")
print(f"   ✓ Optimal Threshold: {optimal_threshold:.4f}")
print("   ✓ Analyzed 5 different threshold values")

print("\n4️⃣ Error Analysis:")
print(f"   ✓ False Positives: {fp_count:,}")
print(f"   ✓ False Negatives: {fn_count:,}")
print("   ✓ Confidence distribution analyzed")

print("\n5️⃣ Per-Class Performance:")
print(f"   ✓ Best Class: {best_class_idx} (F1={df_report.loc[best_class_idx, 'f1-score']:.4f})")
print(f"   ✓ Worst Class: {worst_class_idx} (F1={df_report.loc[worst_class_idx, 'f1-score']:.4f})")

print("\n6️⃣ Advanced Metrics:")
print(f"   ✓ Cohen's Kappa (Binary): {kappa_binary:.4f}")
print(f"   ✓ Matthews Correlation: {mcc_binary:.4f}")
print(f"   ✓ Cohen's Kappa (Multi): {kappa_multi:.4f}")

print("\n7️⃣ Learning Curve Analysis:")
print(f"   ✓ Binary Model - Generalization Gap: {abs(final_train_acc_binary - final_val_acc_binary):.4f}")
print(f"   ✓ Multi-Class - Generalization Gap: {abs(final_train_acc_multi - final_val_acc_multi):.4f}")

print("\n📊 KEY INSIGHTS:")
print("="*70)
print("  🔵 Binary Model:")
print("     • Strong performance for attack detection")
print("     • High precision, moderate recall")
print(f"     • Excellent AUC score ({roc_auc:.4f})")

print("\n  🟠 Multi-Class Model:")
print("     • Challenges with rare attack types (R2L, U2R)")
print("     • Good performance on DoS and Normal classes")
print("     • Class imbalance impacts performance")

print("\n💡 RECOMMENDATIONS:")
print("  1. Consider class weighting for multi-class model")
print("  2. Explore ensemble methods")
print("  3. Augment data for rare attack types")
print("  4. Fine-tune threshold based on cost matrix")

print("\n" + "="*70)
print("✅ MODULE 4 COMPLETED SUCCESSFULLY!")
print("🚀 Ready for Module 5: Deployment")
print("="*70)

## 💾 14. Save Evaluation Results

In [ ]:
# Save evaluation metrics for documentation
evaluation_results = {
    'binary_metrics': {
        'accuracy': float(accuracy_score(y_test_binary, y_pred_binary)),
        'precision': float(precision_score(y_test_binary, y_pred_binary)),
        'recall': float(recall_score(y_test_binary, y_pred_binary)),
        'f1': float(f1_score(y_test_binary, y_pred_binary)),
        'roc_auc': float(roc_auc),
        'kappa': float(kappa_binary),
        'mcc': float(mcc_binary)
    },
    'multiclass_metrics': {
        'accuracy': float(accuracy_score(y_test_multi, y_pred_multi)),
        'precision_weighted': float(precision_score(y_test_multi, y_pred_multi, average='weighted')),
        'recall_weighted': float(recall_score(y_test_multi, y_pred_multi, average='weighted')),
        'f1_weighted': float(f1_score(y_test_multi, y_pred_multi, average='weighted')),
        'macro_auc': float(np.mean(list(roc_auc_multi.values()))),
        'kappa': float(kappa_multi)
    }
}

import json
with open('/content/evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

# Save comparison DataFrame
df_comparison.to_csv('/content/model_comparison.csv', index=False)

print("\n💾 Evaluation Results Saved:")
print("="*70)
print("✅ evaluation_results.json")
print("✅ model_comparison.csv")
print("\n🚀 Ready for Module 5: Deployment!")